# 4.20 — UMAP

UMAP is a way to turn unlabeled high-dimensional data into a small map by deciding which points are near each other, turning those decisions into a weighted graph, and reading geometry from that graph. In this lesson we build the graph-and-Laplacian core from scratch with NumPy, because the same math explains why scaling, neighbor choices, disconnected components, and stability checks matter.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build UMAP's graph view one idea at a time. Run each cell in order and inspect every small matrix. The walkthrough is self-contained and uses a `_w` suffix so it never clashes with the examples below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

### 1. Data become geometry only after we choose a distance

UMAP starts from measured coordinates, but the algorithm never sees a label or a human story. It sees distances. That makes scaling a modeling decision: a feature with larger units can dominate Euclidean distance and therefore dominate the graph.

In [ ]:
X_w = np.array([[0.0, 0.0], [0.0, 1.0], [4.0, 0.0], [4.0, 1.0]])
labels_w = ["A", "B", "C", "D"]
print("X shape:", X_w.shape)
print(X_w)

▶ What you'll see: four points in two measured coordinates, arranged as two vertical pairs.

In [ ]:
diff_w = X_w[:, None, :] - X_w[None, :, :]
D_w = np.sqrt(np.sum(diff_w ** 2, axis=2))
print("pairwise distances:\n", np.round(D_w, 3))
assert D_w.shape == (4, 4)

▶ What you'll see: nearby vertical neighbors are distance 1, while cross-pair neighbors are distance 4 or more.

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(X_w[:, 0], X_w[:, 1], s=90, color="teal")
for i_w, name_w in enumerate(labels_w):
    plt.text(X_w[i_w, 0] + 0.06, X_w[i_w, 1] + 0.03, name_w)
plt.title("1: data as points before graph building")
plt.xlabel("feature 0")
plt.ylabel("feature 1")
plt.show()

▶ What you'll see: the eye sees two close pairs, but UMAP will only know that through distances.

*Why it's done this way:* a distance matrix is the audit trail for the rest of the method. If one feature has a huge numeric scale, its squared differences dominate the sum inside Euclidean distance, so the neighbor graph becomes a scale artifact rather than a manifold clue.

### 2. Distances become a weighted affinity graph

The next idea is to replace all pairwise distances with a local graph. For each point we keep its nearest neighbor and store an affinity weight. A simple Gaussian weight, $\exp(-d^2/\sigma^2)$, makes closer points heavier and farther points lighter.

In [ ]:
sigma_w = 1.5
A_w = np.zeros((4, 4))
for i_w in range(4):
    order_w = np.argsort(D_w[i_w])
    nbr_w = order_w[1]
    A_w[i_w, nbr_w] = np.exp(-(D_w[i_w, nbr_w] ** 2) / sigma_w ** 2)
A_w = np.maximum(A_w, A_w.T)
print("affinity graph A:\n", np.round(A_w, 3))

▶ What you'll see: A connects A-B and C-D with equal positive weights, and leaves the cross-pair edges at zero.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(A_w, cmap="viridis")
plt.colorbar(label="affinity")
plt.xticks(range(4), labels_w)
plt.yticks(range(4), labels_w)
plt.title("2: weighted nearest-neighbor graph A")
plt.show()

▶ What you'll see: two bright off-diagonal blocks and dark cross-block entries.

*Why it's done this way:* the graph is the algorithm's definition of local structure. Keeping local edges protects small neighborhoods from being washed out by global distances, while the exponential weight keeps the graph differentiable in spirit: a distance change becomes a smooth weight change.

### 3. Degrees summarize how much graph mass touches each point

Once we have affinities, the degree of a point is the sum of the weights connected to it. The diagonal degree matrix $D$ is bookkeeping, but it is essential because a highly connected point should not be treated the same as an isolated one.

In [ ]:
deg_w = A_w.sum(axis=1)
Deg_w = np.diag(deg_w)
print("degrees:", np.round(deg_w, 3))
print("D matrix:\n", np.round(Deg_w, 3))
assert np.allclose(deg_w, deg_w[0])

▶ What you'll see: all four degrees match because this toy graph contains two identical two-node components.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(labels_w, deg_w, color="orange")
plt.title("3: graph degree per point")
plt.ylabel("sum of incident affinities")
plt.show()

▶ What you'll see: every point has the same graph mass in this symmetric example.

*Why it's done this way:* degrees normalize the raw adjacency information into conservation accounting. The Laplacian subtracts adjacency from degree, so each row measures how different a point is from the weighted average of its graph neighbors.

### 4. The Laplacian turns a graph into a smoothness operator

The unnormalized graph Laplacian is $L=D-A$. It is small on vectors that assign similar values to connected points and large on vectors that jump across strong edges. This is the bridge from graph structure to coordinates.

In [ ]:
L_w = Deg_w - A_w
print("Laplacian L = D - A:\n", np.round(L_w, 3))
row_sums_w = L_w.sum(axis=1)
print("row sums:", np.round(row_sums_w, 6))
assert np.allclose(row_sums_w, 0.0)

▶ What you'll see: each row sums to zero, which is why a constant vector is always a zero-eigenvalue direction.

In [ ]:
z_w = np.array([1.0, 1.0, -1.0, -1.0])
energy_w = float(z_w @ L_w @ z_w)
print("smoothness energy z^T L z:", round(energy_w, 3))
assert round(energy_w, 3) == 0.0

▶ What you'll see: assigning one constant value per disconnected component costs zero energy.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(L_w, cmap="coolwarm")
plt.colorbar(label="L value")
plt.xticks(range(4), labels_w)
plt.yticks(range(4), labels_w)
plt.title("4: Laplacian matrix")
plt.show()

▶ What you'll see: positive diagonal mass is balanced by negative neighbor entries.

*Why it's done this way:* $z^T L z$ equals a weighted sum of squared differences across edges. Minimizing that energy gives coordinates that vary slowly along strong graph edges, exactly the notion of preserving local neighborhoods.

### 5. Eigenvectors become low-dimensional coordinates

Solving $Lv=\lambda v$ finds directions ordered by graph smoothness. The smallest eigenvalue is zero for the constant vector. The next nontrivial eigenvectors provide coordinates that separate graph components or slowly varying manifold directions.

In [ ]:
evals_w, evecs_w = np.linalg.eigh(L_w)
print("eigenvalues:", np.round(evals_w, 3))
zero_count_w = int(np.sum(np.isclose(evals_w, 0.0, atol=1e-8)))
print("zero eigenvalues:", zero_count_w)
assert np.allclose(np.round(evals_w, 3), [0.0, 0.0, round(2 * deg_w[0], 3), round(2 * deg_w[0], 3)])

▶ What you'll see: two zero eigenvalues, matching the two disconnected graph components.

In [ ]:
Z_w = evecs_w[:, :2]
print("first two eigenvector coordinates:\n", np.round(Z_w, 3))
print("Z shape:", Z_w.shape)
assert Z_w.shape == (4, 2)

▶ What you'll see: a two-column representation for four points, with one zero direction per component.

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(Z_w[:, 0], Z_w[:, 1], s=90, color="seagreen")
for i_w, name_w in enumerate(labels_w):
    plt.text(Z_w[i_w, 0] + 0.02, Z_w[i_w, 1] + 0.02, name_w)
plt.title("5: eigenvectors as graph coordinates")
plt.xlabel("eigenvector 0")
plt.ylabel("eigenvector 1")
plt.show()

▶ What you'll see: points in the same connected component share the same zero-energy coordinate pattern.

*Why it's done this way:* eigenvectors solve the constrained smoothness problem exactly for this linear graph objective. Small eigenvalues mean a coordinate can change without crossing strong edges, so they reveal components and broad geometry before noisy high-frequency directions.

### 6. Hyperparameters are lenses, so stability must be checked

Changing the neighbor count changes the graph, and changing the graph changes the embedding. UMAP outputs should therefore be read as a lens on structure, not as ground truth. A quick stability check compares nearby choices.

In [ ]:
D2_w = D_w.copy()
np.fill_diagonal(D2_w, np.inf)
A1_w = np.zeros((4, 4))
A2_w = np.zeros((4, 4))
for i_w in range(4):
    one_w = np.argsort(D2_w[i_w])[:1]
    two_w = np.argsort(D2_w[i_w])[:2]
    A1_w[i_w, one_w] = 1.0
    A2_w[i_w, two_w] = 1.0
A1_w = np.maximum(A1_w, A1_w.T)
A2_w = np.maximum(A2_w, A2_w.T)
print("edges with k=1:", int(A1_w.sum() / 2), "edges with k=2:", int(A2_w.sum() / 2))

▶ What you'll see: the larger neighborhood adds cross-pair edges, so the graph lens changes.

In [ ]:
L1_w = np.diag(A1_w.sum(axis=1)) - A1_w
L2_w = np.diag(A2_w.sum(axis=1)) - A2_w
e1_w = np.linalg.eigvalsh(L1_w)
e2_w = np.linalg.eigvalsh(L2_w)
print("k=1 eigenvalues:", np.round(e1_w, 3))
print("k=2 eigenvalues:", np.round(e2_w, 3))
assert int(np.sum(np.isclose(e1_w, 0.0))) == 2
assert int(np.sum(np.isclose(e2_w, 0.0))) == 1

▶ What you'll see: k=1 has two components, while k=2 connects the whole graph into one component.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(np.sort(e1_w), marker="o", label="k=1")
plt.plot(np.sort(e2_w), marker="s", label="k=2")
plt.title("6: spectrum changes with neighborhood size")
plt.xlabel("eigenvalue index")
plt.ylabel("eigenvalue")
plt.legend()
plt.show()

▶ What you'll see: the number of near-zero eigenvalues changes when the graph becomes connected.

*Why it's done this way:* stability checks ask whether the discovered structure survives nearby modeling choices. If a tiny change in k flips components or coordinates, the map may be a fragile artifact of the criterion rather than a robust pattern in the data.

## 🛠️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

def pairwise_distances(X):
    diff = X[:, None, :] - X[None, :, :]
    return np.sqrt(np.sum(diff ** 2, axis=2))

def knn_affinity(X, k=1, sigma=1.0):
    D = pairwise_distances(X)
    A = np.zeros((X.shape[0], X.shape[0]))
    masked = D.copy()
    np.fill_diagonal(masked, np.inf)
    for i in range(X.shape[0]):
        nbrs = np.argsort(masked[i])[:k]
        A[i, nbrs] = np.exp(-(D[i, nbrs] ** 2) / (sigma ** 2))
    return np.maximum(A, A.T)

def laplacian(A):
    return np.diag(A.sum(axis=1)) - A

def spectral_coordinates(A, dim=2):
    vals, vecs = np.linalg.eigh(laplacian(A))
    return vals, vecs[:, :dim]

## 🟢 Basics (warm-up)

### Basic 1 — Make a tiny unlabeled dataset

**Goal.** Create points with no labels used by the algorithm, because UMAP-style learning starts from geometry rather than targets. We build it in 2 steps.

In [ ]:
X_b1 = np.array([[0.0, 0.0], [0.2, 0.1], [3.0, 0.0], [3.2, 0.1]])
print("shape:", X_b1.shape)
print(X_b1)
assert X_b1.shape == (4, 2)

▶ What you'll see: a 4×2 matrix, meaning four examples and two measured coordinates.

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(X_b1[:, 0], X_b1[:, 1], s=90, color="teal")
plt.title("Basic 1: unlabeled points")
plt.xlabel("feature 0")
plt.ylabel("feature 1")
plt.show()

▶ What you'll see: two visible pairs, but no class labels are provided to the code.

👀 Takeaway: UMAP begins with coordinates and a distance choice, not a teacher signal.

### Basic 2 — Compute pairwise distances

**Goal.** Build the full distance matrix, because neighborhood graphs are chosen from pairwise closeness. We build it in 2 steps.

In [ ]:
X_b2 = np.array([[0.0, 0.0], [0.0, 1.0], [4.0, 0.0], [4.0, 1.0]])
D_b2 = pairwise_distances(X_b2)
print("distances:\n", np.round(D_b2, 3))
assert round(float(D_b2[0, 1]), 3) == 1.0

▶ What you'll see: the matrix is symmetric and has zeros on the diagonal.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(D_b2, cmap="magma")
plt.colorbar(label="distance")
plt.title("Basic 2: pairwise distances")
plt.show()

▶ What you'll see: small distances are concentrated within the two close pairs.

👀 Takeaway: every later graph decision is downstream of this numeric distance table.

### Basic 3 — Find one nearest neighbor per point

**Goal.** Select local neighbors, because UMAP preserves local neighborhoods rather than every long-range distance. We build it in 2 steps.

In [ ]:
D_b3 = pairwise_distances(np.array([[0.0, 0.0], [0.0, 1.0], [4.0, 0.0], [4.0, 1.0]]))
masked_b3 = D_b3.copy()
np.fill_diagonal(masked_b3, np.inf)
nn_b3 = np.argmin(masked_b3, axis=1)
print("nearest neighbor index per point:", nn_b3)
assert np.array_equal(nn_b3, np.array([1, 0, 3, 2]))

▶ What you'll see: each point chooses the other point in its close vertical pair.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(np.arange(4), masked_b3[np.arange(4), nn_b3], color="orange")
plt.title("Basic 3: nearest-neighbor distances")
plt.xlabel("point index")
plt.ylabel("distance to nearest neighbor")
plt.show()

▶ What you'll see: all nearest-neighbor distances equal 1 in this symmetric toy case.

👀 Takeaway: the neighbor rule converts raw distances into local evidence.

### Basic 4 — Convert a distance to an affinity weight

**Goal.** Use an exponential kernel, because a smooth weight lets close points count more than far points. We build it in 2 steps.

In [ ]:
d_b4 = np.array([0.0, 1.0, 2.0, 4.0])
sigma_b4 = 2.0
w_b4 = np.exp(-(d_b4 ** 2) / sigma_b4 ** 2)
print("weights:", np.round(w_b4, 3))
assert round(float(w_b4[1]), 3) == 0.779

▶ What you'll see: weight is 1 at distance 0 and shrinks as distance grows.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(d_b4, w_b4, marker="o", color="seagreen")
plt.title("Basic 4: distance-to-weight curve")
plt.xlabel("distance")
plt.ylabel("affinity")
plt.show()

▶ What you'll see: far distances fade toward zero influence.

👀 Takeaway: affinities are soft neighbor strengths, not just yes-or-no edges.

### Basic 5 — Build a symmetric affinity matrix

**Goal.** Assemble the graph matrix A, because graph methods operate on point-to-point weights. We build it in 2 steps.

In [ ]:
X_b5 = np.array([[0.0, 0.0], [0.0, 1.0], [4.0, 0.0], [4.0, 1.0]])
A_b5 = knn_affinity(X_b5, k=1, sigma=2.0)
print("A:\n", np.round(A_b5, 3))
assert np.allclose(A_b5, A_b5.T)

▶ What you'll see: the graph has two undirected edges with equal weights.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(A_b5, cmap="viridis")
plt.colorbar(label="affinity")
plt.title("Basic 5: symmetric affinity graph")
plt.show()

▶ What you'll see: the matrix has mirrored bright entries because the graph is undirected.

👀 Takeaway: symmetrizing makes either point's neighbor choice count as a shared graph edge.

### Basic 6 — Compute degrees

**Goal.** Sum graph weights at each point, because degrees tell the Laplacian how much mass each node owns. We build it in 2 steps.

In [ ]:
A_b6 = knn_affinity(np.array([[0.0, 0.0], [0.0, 1.0], [4.0, 0.0], [4.0, 1.0]]), k=1, sigma=2.0)
deg_b6 = A_b6.sum(axis=1)
print("degrees:", np.round(deg_b6, 3))
assert np.allclose(deg_b6, deg_b6[0])

▶ What you'll see: each point has one edge of the same strength.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["0", "1", "2", "3"], deg_b6, color="purple")
plt.title("Basic 6: degree per point")
plt.ylabel("weighted degree")
plt.show()

▶ What you'll see: equal bars because the toy graph is balanced.

👀 Takeaway: degree is the local normalizer behind the Laplacian.

### Basic 7 — Form the Laplacian

**Goal.** Compute L = D - A, because this matrix measures graph smoothness. We build it in 2 steps.

In [ ]:
A_b7 = knn_affinity(np.array([[0.0, 0.0], [0.0, 1.0], [4.0, 0.0], [4.0, 1.0]]), k=1, sigma=2.0)
L_b7 = laplacian(A_b7)
print("L:\n", np.round(L_b7, 3))
assert np.allclose(L_b7.sum(axis=1), 0.0)

▶ What you'll see: positive diagonal entries and negative neighbor entries balance to row sum zero.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(L_b7, cmap="coolwarm")
plt.colorbar(label="L value")
plt.title("Basic 7: graph Laplacian")
plt.show()

▶ What you'll see: each component appears as a small Laplacian block.

👀 Takeaway: the Laplacian turns graph edges into a matrix that penalizes rough coordinates.

### Basic 8 — Count connected components with eigenvalues

**Goal.** Inspect Laplacian eigenvalues, because zero eigenvalues count disconnected components. We build it in 2 steps.

In [ ]:
A_b8 = knn_affinity(np.array([[0.0, 0.0], [0.0, 1.0], [4.0, 0.0], [4.0, 1.0]]), k=1, sigma=2.0)
vals_b8 = np.linalg.eigvalsh(laplacian(A_b8))
zero_count_b8 = int(np.sum(np.isclose(vals_b8, 0.0, atol=1e-8)))
print("eigenvalues:", np.round(vals_b8, 3), "zero count:", zero_count_b8)
assert zero_count_b8 == 2

▶ What you'll see: two zero eigenvalues reveal two disconnected graph components.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(vals_b8, marker="o", color="navy")
plt.title("Basic 8: Laplacian spectrum")
plt.xlabel("index")
plt.ylabel("eigenvalue")
plt.show()

▶ What you'll see: the spectrum begins with two zeros, then jumps upward.

👀 Takeaway: eigenvalues make graph connectivity measurable.

### Basic 9 — Use eigenvectors as coordinates

**Goal.** Extract low-dimensional graph coordinates, because smooth eigenvectors preserve strong local edges. We build it in 2 steps.

In [ ]:
A_b9 = knn_affinity(np.array([[0.0, 0.0], [0.0, 1.0], [4.0, 0.0], [4.0, 1.0]]), k=1, sigma=2.0)
vals_b9, Z_b9 = spectral_coordinates(A_b9, dim=2)
print("Z:\n", np.round(Z_b9, 3))
assert Z_b9.shape == (4, 2)

▶ What you'll see: four rows of two graph-derived coordinates.

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(Z_b9[:, 0], Z_b9[:, 1], s=90, color="crimson")
for i_b9 in range(4):
    plt.text(Z_b9[i_b9, 0] + 0.02, Z_b9[i_b9, 1] + 0.02, str(i_b9))
plt.title("Basic 9: eigenvector coordinates")
plt.xlabel("coord 0")
plt.ylabel("coord 1")
plt.show()

▶ What you'll see: points in the same component share coordinate structure.

👀 Takeaway: graph eigenvectors are a simple from-scratch embedding mechanism.

### Basic 10 — Check shape bookkeeping

**Goal.** Track matrix shapes, because most implementation bugs in embedding code are shape mismatches. We build it in 2 steps.

In [ ]:
X_b10 = np.array([[0.0, 0.0], [0.0, 1.0], [4.0, 0.0], [4.0, 1.0]])
A_b10 = knn_affinity(X_b10, k=1, sigma=2.0)
L_b10 = laplacian(A_b10)
vals_b10, Z_b10 = spectral_coordinates(A_b10, dim=1)
print("X", X_b10.shape, "A", A_b10.shape, "L", L_b10.shape, "Z", Z_b10.shape)
assert X_b10.shape == (4, 2) and Z_b10.shape == (4, 1)

▶ What you'll see: data are 4×2, graph matrices are 4×4, and the embedding is 4×1.

In [ ]:
plt.figure(figsize=(4, 2.5))
plt.scatter(Z_b10[:, 0], np.zeros(4), s=90, color="teal")
plt.yticks([])
plt.title("Basic 10: one-dimensional representation")
plt.xlabel("Z coordinate")
plt.show()

▶ What you'll see: the embedding keeps one coordinate per example.

👀 Takeaway: UMAP-like workflows transform example features into example embeddings, while graph matrices stay example by example.

## 🟡 Easy

### Easy 1 — Scaling can change neighbors

**Goal.** Show scale sensitivity, because distance-based graphs only see the numeric units we provide. We build it in 3 steps.

In [ ]:
X_e1 = np.array([[0.0, 0.0], [0.0, 2.0], [1.0, 0.1]])
D_raw_e1 = pairwise_distances(X_e1)
print("raw distances from point 0:", np.round(D_raw_e1[0], 3))

▶ What you'll see: point 2 is closer to point 0 than point 1 under the raw coordinates.

In [ ]:
X_scaled_e1 = X_e1.copy()
X_scaled_e1[:, 0] *= 5.0
D_scaled_e1 = pairwise_distances(X_scaled_e1)
print("scaled distances from point 0:", np.round(D_scaled_e1[0], 3))
assert int(np.argmin(np.where(np.arange(3) == 0, np.inf, D_raw_e1[0]))) == 2
assert int(np.argmin(np.where(np.arange(3) == 0, np.inf, D_scaled_e1[0]))) == 1

▶ What you'll see: after scaling feature 0, the nearest neighbor of point 0 flips.

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(X_e1[:, 0], X_e1[:, 1], s=90, label="raw")
plt.scatter(X_scaled_e1[:, 0], X_scaled_e1[:, 1], s=90, marker="x", label="scaled")
plt.title("Easy 1: feature scale changes geometry")
plt.xlabel("feature 0")
plt.ylabel("feature 1")
plt.legend()
plt.show()

▶ What you'll see: stretching one axis changes which point is geometrically close.

👀 Takeaway: scaling is preprocessing, but it is also a modeling choice for UMAP.

### Easy 2 — Compare k-neighborhood graphs

**Goal.** Build graphs with k=1 and k=2, because neighborhood size controls local versus global connectivity. We build it in 3 steps.

In [ ]:
X_e2 = np.array([[0.0, 0.0], [0.0, 1.0], [4.0, 0.0], [4.0, 1.0]])
A1_e2 = knn_affinity(X_e2, k=1, sigma=3.0)
A2_e2 = knn_affinity(X_e2, k=2, sigma=3.0)
print("edge counts:", int(np.sum(A1_e2 > 0) / 2), int(np.sum(A2_e2 > 0) / 2))

▶ What you'll see: k=2 has more graph edges than k=1.

In [ ]:
z1_e2 = int(np.sum(np.isclose(np.linalg.eigvalsh(laplacian(A1_e2)), 0.0, atol=1e-8)))
z2_e2 = int(np.sum(np.isclose(np.linalg.eigvalsh(laplacian(A2_e2)), 0.0, atol=1e-8)))
print("component counts:", z1_e2, z2_e2)
assert z1_e2 == 2 and z2_e2 == 1

▶ What you'll see: k=2 connects the two pairs into a single graph.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].imshow(A1_e2, cmap="viridis")
ax[0].set_title("k=1")
ax[1].imshow(A2_e2, cmap="viridis")
ax[1].set_title("k=2")
plt.suptitle("Easy 2: neighborhood size changes A")
plt.show()

▶ What you'll see: the k=2 matrix has extra bright cross-pair entries.

👀 Takeaway: k is a lens width, and the graph spectrum reveals its connectivity effect.

### Easy 3 — Make a one-dimensional spectral map

**Goal.** Embed a connected chain into one coordinate, because the second-smallest eigenvector often orders points along a graph. We build it in 3 steps.

In [ ]:
X_e3 = np.array([[0.0], [1.0], [2.0], [3.0], [4.0]])
A_e3 = knn_affinity(X_e3, k=2, sigma=2.0)
vals_e3, vecs_e3 = np.linalg.eigh(laplacian(A_e3))
coord_e3 = vecs_e3[:, 1]
print("eigenvalues:", np.round(vals_e3, 3))
assert vals_e3[1] > 0

▶ What you'll see: one zero eigenvalue followed by a small positive smooth direction.

In [ ]:
print("1D coordinate:", np.round(coord_e3, 3))
order_e3 = np.argsort(coord_e3)
print("coordinate order:", order_e3)
assert set(order_e3.tolist()) == set(range(5))

▶ What you'll see: the eigenvector gives a monotone ordering up to sign.

In [ ]:
plt.figure(figsize=(5, 3))
plt.scatter(coord_e3, np.zeros_like(coord_e3), s=90, color="seagreen")
for i_e3 in range(5):
    plt.text(coord_e3[i_e3] + 0.01, 0.01, str(i_e3))
plt.yticks([])
plt.title("Easy 3: chain embedded in 1D")
plt.xlabel("spectral coordinate")
plt.show()

▶ What you'll see: neighboring chain points stay near each other in the coordinate.

👀 Takeaway: smooth graph eigenvectors can recover a simple manifold order.

### Easy 4 — Detect disconnected components

**Goal.** Count graph components from zero eigenvalues, because disconnected pieces cannot be smoothly arranged by a single connected coordinate. We build it in 3 steps.

In [ ]:
A_e4 = np.array([[0.0, 1.0, 0.0, 0.0, 0.0],
                 [1.0, 0.0, 0.0, 0.0, 0.0],
                 [0.0, 0.0, 0.0, 1.0, 0.0],
                 [0.0, 0.0, 1.0, 0.0, 1.0],
                 [0.0, 0.0, 0.0, 1.0, 0.0]])
L_e4 = laplacian(A_e4)
vals_e4 = np.linalg.eigvalsh(L_e4)
print("eigenvalues:", np.round(vals_e4, 3))

▶ What you'll see: the graph has two separate blocks and therefore two zero eigenvalues.

In [ ]:
components_e4 = int(np.sum(np.isclose(vals_e4, 0.0, atol=1e-8)))
print("components:", components_e4)
assert components_e4 == 2

▶ What you'll see: the component count is read directly from the Laplacian spectrum.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(A_e4, cmap="Greens")
plt.title("Easy 4: two graph components")
plt.colorbar(label="edge")
plt.show()

▶ What you'll see: two disconnected adjacency blocks.

👀 Takeaway: zero eigenvalues are not numerical decoration; they diagnose disconnected structure.

### Easy 5 — Quantify graph smoothness

**Goal.** Compute $z^T L z$, because embeddings prefer coordinates that do not jump across strong edges. We build it in 3 steps.

In [ ]:
A_e5 = np.array([[0.0, 1.0, 0.0], [1.0, 0.0, 1.0], [0.0, 1.0, 0.0]])
L_e5 = laplacian(A_e5)
z_smooth_e5 = np.array([1.0, 1.0, 1.0])
z_rough_e5 = np.array([1.0, -1.0, 1.0])
print("L:\n", L_e5)

▶ What you'll see: the middle point connects to both endpoints.

In [ ]:
energy_smooth_e5 = float(z_smooth_e5 @ L_e5 @ z_smooth_e5)
energy_rough_e5 = float(z_rough_e5 @ L_e5 @ z_rough_e5)
print("energies:", energy_smooth_e5, energy_rough_e5)
assert energy_smooth_e5 == 0.0 and energy_rough_e5 == 8.0

▶ What you'll see: a constant coordinate costs zero, while a sign flip across edges costs a lot.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["smooth", "rough"], [energy_smooth_e5, energy_rough_e5], color=["teal", "red"])
plt.title("Easy 5: Laplacian smoothness energy")
plt.ylabel("z^T L z")
plt.show()

▶ What you'll see: the rough assignment has much higher energy.

👀 Takeaway: spectral embeddings keep connected neighbors close by minimizing graph roughness.

## 🔴 Advanced

### Advanced 1 — Sweep sigma in the affinity kernel

**Goal.** Compare kernel widths, because sigma controls how quickly distance fades into weak affinity. We build it in 3 steps.

In [ ]:
X_a1 = np.array([[0.0], [1.0], [2.0], [5.0]])
sigmas_a1 = np.array([0.5, 1.0, 2.0, 4.0])
weights_a1 = []
for sigma_a1 in sigmas_a1:
    A_a1 = knn_affinity(X_a1, k=2, sigma=sigma_a1)
    weights_a1.append(A_a1[0, 1])
print("edge 0-1 weights:", np.round(weights_a1, 3))

▶ What you'll see: the same distance receives larger weight as sigma grows.

In [ ]:
lambda2_a1 = []
for sigma_a1 in sigmas_a1:
    vals_a1 = np.linalg.eigvalsh(laplacian(knn_affinity(X_a1, k=2, sigma=sigma_a1)))
    lambda2_a1.append(vals_a1[1])
print("second eigenvalues:", np.round(lambda2_a1, 3))
assert len(lambda2_a1) == 4

▶ What you'll see: the graph's low-frequency spectrum changes as affinities soften.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(sigmas_a1, lambda2_a1, marker="o", color="purple")
plt.title("Advanced 1: sigma changes graph connectivity strength")
plt.xlabel("sigma")
plt.ylabel("second-smallest eigenvalue")
plt.show()

▶ What you'll see: larger sigma generally strengthens weak long edges and changes spectral gaps.

👀 Takeaway: sigma is a locality knob, so it should be inspected rather than accepted blindly.

### Advanced 2 — Compare raw and standardized embeddings

**Goal.** Standardize features before graph building, because unscaled features can dominate distances and distort the map. We build it in 4 steps.

In [ ]:
X_a2 = np.array([[0.0, 0.0], [0.0, 2.0], [1.0, 0.2], [1.0, 2.2]])
X_big_a2 = X_a2.copy()
X_big_a2[:, 0] *= 10.0
A_raw_a2 = knn_affinity(X_big_a2, k=1, sigma=3.0)
print("raw scaled A edges:", int(np.sum(A_raw_a2 > 0) / 2))

▶ What you'll see: the graph is built after feature 0 has been artificially magnified.

In [ ]:
mean_a2 = X_big_a2.mean(axis=0)
std_a2 = X_big_a2.std(axis=0)
X_std_a2 = (X_big_a2 - mean_a2) / std_a2
A_std_a2 = knn_affinity(X_std_a2, k=1, sigma=3.0)
print("standardized feature means:", np.round(X_std_a2.mean(axis=0), 6))
assert np.allclose(X_std_a2.mean(axis=0), 0.0)

▶ What you'll see: standardization centers both features and puts them on comparable units.

In [ ]:
vals_raw_a2, Z_raw_a2 = spectral_coordinates(A_raw_a2, dim=2)
vals_std_a2, Z_std_a2 = spectral_coordinates(A_std_a2, dim=2)
print("raw eigenvalues:", np.round(vals_raw_a2, 3))
print("std eigenvalues:", np.round(vals_std_a2, 3))

▶ What you'll see: changing scale can change graph weights and therefore the spectral coordinates.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].scatter(Z_raw_a2[:, 0], Z_raw_a2[:, 1], s=90, color="red")
ax[0].set_title("raw scaled")
ax[1].scatter(Z_std_a2[:, 0], Z_std_a2[:, 1], s=90, color="teal")
ax[1].set_title("standardized")
plt.suptitle("Advanced 2: preprocessing changes the map")
plt.show()

▶ What you'll see: the two embeddings can organize points differently because their graphs differ.

👀 Takeaway: preprocessing is part of the unsupervised model, not a neutral prelude.

### Advanced 3 — Align two embeddings before comparing stability

**Goal.** Compare embeddings up to sign and rotation, because eigenvector coordinates are not unique in orientation. We build it in 4 steps.

In [ ]:
X_a3 = np.array([[0.0], [1.0], [2.0], [3.0], [4.0]])
A_a3 = knn_affinity(X_a3, k=2, sigma=2.0)
_, Z_a3 = spectral_coordinates(A_a3, dim=2)
Z_flip_a3 = Z_a3 @ np.array([[0.0, 1.0], [1.0, 0.0]])
print("original shape:", Z_a3.shape, "flipped shape:", Z_flip_a3.shape)
assert Z_a3.shape == Z_flip_a3.shape

▶ What you'll see: both embeddings have the same information but swapped coordinate axes.

In [ ]:
U_a3, _, Vt_a3 = np.linalg.svd(Z_flip_a3.T @ Z_a3)
R_a3 = U_a3 @ Vt_a3
Z_aligned_a3 = Z_flip_a3 @ R_a3
err_a3 = float(np.linalg.norm(Z_aligned_a3 - Z_a3))
print("alignment error:", round(err_a3, 6))
assert err_a3 < 1e-10

▶ What you'll see: after orthogonal alignment, the swapped embedding matches the original.

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(Z_a3[:, 0], Z_a3[:, 1], s=90, label="original")
plt.scatter(Z_aligned_a3[:, 0], Z_aligned_a3[:, 1], marker="x", s=90, label="aligned")
plt.title("Advanced 3: compare after alignment")
plt.legend()
plt.show()

▶ What you'll see: the aligned crosses sit on top of the original points.

👀 Takeaway: stability checks should compare geometry, not arbitrary eigenvector orientation.

### Advanced 4 — Bootstrap a graph stability score

**Goal.** Resample points and compare neighbor edges, because fragile graphs produce fragile embeddings. We build it in 4 steps.

In [ ]:
X_a4 = np.array([[0.0, 0.0], [0.1, 0.0], [1.0, 0.0], [1.1, 0.0], [3.0, 0.0], [3.1, 0.0]])
A_ref_a4 = knn_affinity(X_a4, k=1, sigma=1.0) > 0
rng_a4 = np.random.default_rng(0)
print("reference edges:", int(A_ref_a4.sum() / 2))

▶ What you'll see: the reference graph links close pairs.

In [ ]:
scores_a4 = []
for t_a4 in range(20):
    noise_a4 = 0.03 * rng_a4.normal(size=X_a4.shape)
    A_noisy_a4 = knn_affinity(X_a4 + noise_a4, k=1, sigma=1.0) > 0
    both_a4 = np.logical_and(A_ref_a4, A_noisy_a4).sum() / 2
    either_a4 = np.logical_or(A_ref_a4, A_noisy_a4).sum() / 2
    scores_a4.append(both_a4 / either_a4)
print("stability scores:", np.round(scores_a4[:5], 3), "...")
assert min(scores_a4) >= 0.0 and max(scores_a4) <= 1.0

▶ What you'll see: each score is a Jaccard overlap between the original and noisy edge sets.

In [ ]:
mean_score_a4 = float(np.mean(scores_a4))
print("mean stability:", round(mean_score_a4, 3))
assert mean_score_a4 > 0.8

▶ What you'll see: this well-separated toy graph is highly stable to small noise.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(scores_a4, marker="o", color="seagreen")
plt.ylim(0, 1.05)
plt.title("Advanced 4: neighbor-graph stability")
plt.xlabel("noise trial")
plt.ylabel("edge Jaccard score")
plt.show()

▶ What you'll see: scores stay near 1 when local neighbor relationships are robust.

👀 Takeaway: stability checks turn the warning about fragile unsupervised outputs into a measurable diagnostic.

### Advanced 5 — Use a fallback when the graph is disconnected

**Goal.** Detect disconnected graphs before interpreting coordinates, because disconnected components make relative positions arbitrary. We build it in 4 steps.

In [ ]:
X_a5 = np.array([[0.0, 0.0], [0.0, 1.0], [5.0, 0.0], [5.0, 1.0]])
A_sparse_a5 = knn_affinity(X_a5, k=1, sigma=2.0)
vals_sparse_a5 = np.linalg.eigvalsh(laplacian(A_sparse_a5))
components_sparse_a5 = int(np.sum(np.isclose(vals_sparse_a5, 0.0, atol=1e-8)))
print("sparse components:", components_sparse_a5)
assert components_sparse_a5 == 2

▶ What you'll see: the nearest-neighbor graph splits into two components.

In [ ]:
A_connected_a5 = knn_affinity(X_a5, k=2, sigma=4.0)
vals_connected_a5 = np.linalg.eigvalsh(laplacian(A_connected_a5))
components_connected_a5 = int(np.sum(np.isclose(vals_connected_a5, 0.0, atol=1e-8)))
print("connected components:", components_connected_a5)
assert components_connected_a5 == 1

▶ What you'll see: increasing k and sigma creates enough bridge weight to connect the graph.

In [ ]:
_, Z_sparse_a5 = spectral_coordinates(A_sparse_a5, dim=2)
_, Z_connected_a5 = spectral_coordinates(A_connected_a5, dim=2)
print("sparse Z shape:", Z_sparse_a5.shape, "connected Z shape:", Z_connected_a5.shape)
assert Z_sparse_a5.shape == Z_connected_a5.shape == (4, 2)

▶ What you'll see: both produce coordinates, but only the connected graph has a meaningful global relationship.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].scatter(Z_sparse_a5[:, 0], Z_sparse_a5[:, 1], s=90, color="red")
ax[0].set_title("disconnected")
ax[1].scatter(Z_connected_a5[:, 0], Z_connected_a5[:, 1], s=90, color="teal")
ax[1].set_title("connected")
plt.suptitle("Advanced 5: connectivity check before interpretation")
plt.show()

▶ What you'll see: the connected graph gives one shared coordinate system, while disconnected pieces have arbitrary relative placement.

👀 Takeaway: always diagnose components before reading global meaning into an unsupervised map.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

UMAP turns unlabeled data into structure by choosing the right notion of similarity, compression, or surprise.

Part 4 moves from prediction with labels to discovery without labels. Linear algebra meets graph structure: affinities become a Laplacian whose eigenvectors reveal geometry. UMAP normally uses a fuzzy neighbor graph; here we use sklearn's CPU-only spectral/manifold stand-in so no package install is needed.

Save a copy to Drive to edit.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_digits
from sklearn.decomposition import NMF
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap
from sklearn.manifold import LocallyLinearEmbedding
from sklearn.manifold import MDS
from sklearn.manifold import SpectralEmbedding
from sklearn.manifold import TSNE
from sklearn.manifold import trustworthiness
from sklearn.metrics import pairwise_distances
from sklearn.random_projection import GaussianRandomProjection
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

def dimred_ladder():
    """D1..D5 dimensionality-reduction ladder. Returns [(name, X, y), ...] of rising ambient dim.

    2-D toy -> 3-D swiss-roll-ish -> digits(64-D) -> the same with noise dims -> a wide feature set.
    y is a color/label for visualization only.
    """
    rungs = []
    rng = np.random.default_rng(3)

    t = np.linspace(0, 4, 120)
    x1 = np.column_stack([t, 0.5 * t + rng.normal(0, 0.05, 120)])
    rungs.append(("D1 near-1-D line in 2-D", x1, t))

    tt = np.linspace(0, 3 * np.pi, 200)
    x2 = np.column_stack([tt * np.cos(tt), 8 * rng.random(200), tt * np.sin(tt)])
    rungs.append(("D2 swiss-roll (3-D)", x2, tt))

    digits = load_digits()
    rungs.append(("D3 digits (real, 64-D)", digits.data / 16.0, digits.target))

    xn = np.hstack([digits.data / 16.0, rng.normal(0, 1, size=(digits.data.shape[0], 32))])
    rungs.append(("D4 digits + 32 noise dims", xn, digits.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (30-D)", bc.data, bc.target))

    return rungs



def sample_for_embedding(X, y, max_points=500, seed=SEED):
    rng = np.random.default_rng(seed)
    if X.shape[0] <= max_points:
        return X, y
    idx = rng.choice(X.shape[0], size=max_points, replace=False)
    order = np.sort(idx)
    return X[order], y[order]


def standardize_for_geometry(X):
    scaler = StandardScaler()
    return scaler.fit_transform(X)


def nonnegative_for_factorization(X):
    scaler = MinMaxScaler()
    return scaler.fit_transform(X)


def safe_trustworthiness(X, Z):
    neighbors = min(10, max(1, (X.shape[0] - 1) // 3))
    return float(trustworthiness(X, Z, n_neighbors=neighbors))


def plot_ladder_embeddings(results, metric_name):
    fig, axes = plt.subplots(1, len(results), figsize=(17, 3.4))
    for ax, item in zip(axes, results):
        scatter = ax.scatter(item["Z"][:, 0], item["Z"][:, 1], c=item["y"], s=10, cmap="viridis")
        ax.set_title(item["name"].split("(")[0], fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle("2D embeddings across the D1-D5 ladder")
    plt.show()

    fig, ax = plt.subplots(figsize=(6, 3.5))
    xs = np.arange(1, len(results) + 1)
    ys = [item["metric"] for item in results]
    ax.plot(xs, ys, marker="o")
    ax.set_xticks(xs)
    ax.set_xticklabels([f"D{i}" for i in xs])
    ax.set_ylabel(metric_name)
    ax.set_xlabel("dataset rung")
    ax.set_title(f"{metric_name} vs. ladder complexity")
    ax.grid(True, alpha=0.3)
    plt.show()


def preview_ladder(rungs):
    for i, (name, X, y) in enumerate(rungs, start=1):
        unique = np.unique(y)
        label_info = len(unique) if unique.size < 30 else "continuous"
        print(f"D{i}: {name} | X={X.shape} | color/label info={label_info}")
        print(np.round(X[:3, :min(5, X.shape[1])], 3))

## The concept, built once: graph Laplacian audit

The lesson's graph computation is $$L=D-A,\qquad Lv=\lambda v$$. For two disconnected edges, the Laplacian has two zero eigenvalues, revealing two components.

In [ ]:
A = np.array([[0.0, 1.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 1.0, 0.0]])
degrees = A.sum(axis=1)
D = np.diag(degrees)
L = D - A
eigenvalues = np.linalg.eigvalsh(L)
zero_count = int(np.isclose(eigenvalues, 0.0).sum())
print("degrees", degrees)
print("eigenvalues", eigenvalues)
print("zero eigenvalues", zero_count)
assert np.allclose(degrees, [1.0, 1.0, 1.0, 1.0])
assert np.allclose(np.round(eigenvalues, 3), [0.0, 0.0, 2.0, 2.0])
assert zero_count == 2

Now build the reusable method. We do not install `umap-learn`; the CPU-only stand-in is sklearn `SpectralEmbedding`, with `Isomap` as a fallback, preserving the neighbor-graph caveat.

In [ ]:
def method(X, n_neighbors=15, min_dist=0.1, seed=SEED):
    X_scaled = standardize_for_geometry(X)
    neighbors = min(n_neighbors, X_scaled.shape[0] - 1)
    try:
        model = SpectralEmbedding(n_components=2, n_neighbors=neighbors, random_state=seed, affinity="nearest_neighbors")
        Z = model.fit_transform(X_scaled)
    except Exception as exc:
        print("SpectralEmbedding fallback to Isomap", exc)
        model = Isomap(n_components=2, n_neighbors=neighbors)
        Z = model.fit_transform(X_scaled)
    return Z

assert L.shape == (4, 4)

## The dataset ladder

We use the shared F3 dimensionality-reduction ladder: a near-1D toy line, a 3D swiss-roll-style surface, real handwritten digits, digits with added noise dimensions, and a real 30-dimensional breast-cancer feature table. The labels are only colors for visualization; the embedding method does not train on them.

In [ ]:
rungs = dimred_ladder()
preview_ladder(rungs)

## Run the same method across D1-D5

Each rung is standardized or scaled in the same way, embedded into 2D, and scored with trustworthiness. Subsampling is seeded and only bounds future notebook runtime; this build script does not execute the notebook.

In [ ]:
metric_name = "trustworthiness"
results = []
for rung_index, (name, X, y) in enumerate(rungs, start=1):
    X_small, y_small = sample_for_embedding(X, y, max_points=500, seed=SEED + rung_index)
    Z = method(X_small, n_neighbors=15, min_dist=0.1, seed=SEED + rung_index)
    score = safe_trustworthiness(standardize_for_geometry(X_small), Z)
    results.append({"name": name, "X": X_small, "y": y_small, "Z": Z, "metric": score})
    print(f"D{rung_index} | {name:32s} | trustworthiness={score:.3f}")

## Results visualization

The closing figure has two parts: small-multiple embedding panels for D1-D5, then a metric curve as the data become more realistic and higher dimensional.

In [ ]:
plot_ladder_embeddings(results, metric_name)

## Pitfall on the hardest rung: neighbor scale changes the story

UMAP-like graph embeddings depend on the neighbor graph. On D5, tiny neighborhoods can fragment structure and large neighborhoods can smooth it away; the fix is a parameter sweep with trustworthiness.

In [ ]:
name, X, y = rungs[-1]
X_small, y_small = sample_for_embedding(X, y, max_points=500, seed=29)
neighbors_to_try = [3, 8, 15, 30]
sweep = []
for neighbors in neighbors_to_try:
    Z = method(X_small, n_neighbors=neighbors, min_dist=0.1, seed=0)
    score = safe_trustworthiness(standardize_for_geometry(X_small), Z)
    sweep.append((neighbors, score))
print("neighbor sweep", [(n, round(s, 3)) for n, s in sweep])
print("fix: choose a stable neighborhood range, not a single attractive layout")

## Evaluate it + Practice

- Report trustworthiness next to a no-skill baseline such as plotting two raw standardized features or a random 2D map.
- Sanity check that nearby points in the original space still have nearby points in the embedding.
- Ablate the key idea: remove scaling, change the random seed, or use too few neighbors/components and watch the metric move.
- Watch for failure signals: unstable layouts, one feature dominating distances, disconnected neighbor graphs, or a better objective with worse held-out structure.
- Treat labels as a post-hoc audit only; unsupervised methods do not get to train on them.


### Practice

Run the neighbor sweep on D2 and compare with D5.

In [ ]:
# Your code here


### Practice

Replace SpectralEmbedding with Isomap in the method and compare trustworthiness.

In [ ]:
# Your code here


### Practice

Use the D1 graph audit to build a three-component graph and predict the zero eigenvalue count.

In [ ]:
# Your code here
